# Notebook 0 — Fundamentos y setup
### Serie *Capa 1 del embudo AI-first · Ingesta de datos en GCP*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/capa1-ingesta-colab/blob/main/00_fundamentos_setup.ipynb)

> **Sustituye `USUARIO`** en el badge por tu cuenta de GitHub cuando subas el repo.

---

## Qué vas a conseguir aquí

Al terminar este cuaderno sabrás:

1. Qué es la nube, qué es GCP y qué es Colab — con tus propias palabras.
2. Lo mínimo de Python para entender el resto de la serie (variables, diccionarios, funciones, llamar a una API).
3. Qué es una **API REST** y qué es **JSON**, viéndolo en vivo.
4. Qué significa **autenticarse** y por qué nunca escribimos contraseñas en el código.
5. Conectar este Colab con **tu** proyecto de Google Cloud y lanzar tu primer comando real.
6. Consultar un **dataset público** de BigQuery sin haber ingerido nada todavía.
7. Los **4 modos de ingesta** que organizan toda la clase.
8. Dejar tu entorno **verificado** (verde) para los siguientes notebooks.

**Tiempo estimado:** 60-90 min. Ve despacio: este es el cimiento de los 8 cuadernos siguientes.


## 0.1 · ¿Qué es Google Colab?

**Colab** (abreviatura de *Google Colaboratory*) es un servicio **gratuito** de Google que te da un **cuaderno de código que se ejecuta en la nube**, directamente en tu navegador. No instalas nada: abres una pestaña y ya tienes Python, librerías de datos e IA, y las herramientas de Google Cloud (`gcloud`, `bq`) listas para usar.

Por dentro, Colab es una versión alojada de **Jupyter Notebook**, el formato de cuaderno más usado del mundo en ciencia de datos. Lo que hace especial a Colab:

- **Cero instalación**: todo corre en una máquina Linux temporal de Google. Tu portátil solo necesita un navegador.
- **Gratis, incluido hardware potente**: puedes pedir **GPU o TPU** gratis para tareas de IA (no las usaremos en esta serie, pero conviene saberlo).
- **Se comparte como un Google Doc**: mandas un enlace y otra persona ve, copia y ejecuta tu cuaderno.
- **Conectado a Google**: lee de tu Google Drive y, como verás, **se autentica con tu proyecto de Google Cloud** en una sola celda.

### Las dos piezas de un cuaderno

Un cuaderno alterna dos tipos de **celda**:

| Tipo de celda | Contiene | Para qué |
|---|---|---|
| **Texto (Markdown)** | Explicaciones, títulos, tablas, imágenes | Contar *qué* hacemos y *por qué* |
| **Código** | Python (o comandos con `!`) | *Ejecutar* de verdad y ver el resultado |

Una celda de código se ejecuta con **Shift+Enter**. El resultado (un número, una tabla, un gráfico, un error) aparece **justo debajo** de la celda. El **runtime** es la máquina que ejecuta tu código; es **temporal**: si la cierras o pasa un rato inactiva, se borra lo que estaba en memoria (pero no el cuaderno). Eso lo retomamos en 0.1c.

Ejecuta la celda de abajo para confirmar que tu runtime está vivo. 👇


## 0.1b · ¿Para qué se usa Colab? (casos de uso)

Colab se ha vuelto la herramienta por defecto para cualquier trabajo que mezcle **datos, código y explicación**. Los casos más habituales:

1. **Aprender y enseñar a programar / datos / IA** — como esta serie. El alumno *lee, ejecuta y modifica* en el mismo sitio, sin pelearse con instalaciones. Por eso casi todos los cursos de Python, pandas o machine learning vienen en cuadernos Colab.
2. **Explorar y analizar datos** — cargar un CSV o consultar BigQuery, calcular estadísticas y **dibujar gráficos** que aparecen incrustados. Es el día a día de un analista de datos.
3. **Prototipar modelos de IA / machine learning** — entrenar o probar un modelo usando la **GPU gratuita**, sin comprar hardware. Mucha gente prueba modelos de Hugging Face o entrena redes pequeñas aquí.
4. **Automatizar y orquestar servicios en la nube** — **justo lo que haremos**: desde Colab hablamos con Google Cloud (BigQuery, Pub/Sub, Cloud Run…) para construir tuberías de ingesta reales.
5. **Compartir trabajo reproducible** — un informe que el lector puede *re-ejecutar* para verificar los números, en vez de un PDF estático que puede estar desactualizado.
6. **Prototipado rápido sin montar un entorno** — probar una idea, una API o una librería en 2 minutos sin tocar tu sistema.

### Dónde Colab NO es la herramienta

Para ser honestos (y esto es parte del criterio de ingeniería que enseña la clase):

- **No es para producción**: el runtime es temporal y se apaga. Una ingesta que corre cada noche vive en **Cloud Run / Cloud Scheduler / Workflows**, no en un Colab abierto. Colab es donde *aprendemos y desplegamos*; la infraestructura desplegada corre sola en GCP.
- **No es para aplicaciones con interfaz de usuario** ni servicios que deben estar siempre encendidos.
- **No guarda estado para siempre**: si necesitas persistencia, va a BigQuery, a Cloud Storage o a una base de datos — no en variables de Python en memoria.

> En esta serie usamos Colab como **cabina de mando**: desde aquí pensamos, ejecutamos comandos `gcloud` y desplegamos. Lo que desplegamos (un topic de Pub/Sub, un Workflow, un trigger de Eventarc) **sigue corriendo en GCP aunque cierres el Colab**.


## 0.1c · ¿Qué ganamos juntando texto y código así?

Mezclar narrativa y código ejecutable en el mismo documento tiene un nombre — **"literate programming"** (programación legible, una idea de Donald Knuth) — y nos da ventajas muy concretas frente a un script `.py` suelto o un PDF de teoría:

1. **Aprendes el *porqué* pegado al *qué*.** La explicación está justo encima del código que la realiza. No saltas entre un manual y un editor: lo lees y lo ejecutas en el mismo flujo.
2. **Los resultados viven junto al código.** Una tabla, un número o un gráfico aparecen **debajo** de la celda que los produjo. El documento *demuestra* lo que afirma, no solo lo cuenta.
3. **Es reproducible.** Cualquiera puede re-ejecutar el cuaderno y obtener los mismos resultados. En una clase, esto significa que **20 alumnos corren exactamente lo mismo** sin "en mi máquina funcionaba".
4. **La documentación no se queda obsoleta.** En un proyecto normal, los comentarios envejecen y mienten. Aquí, *el código documentado es el código que se ejecuta*: si está mal, falla a la vista.
5. **Invita a experimentar.** Cambias un valor, pulsas Shift+Enter y ves el efecto al instante. Ese bucle *leer → ejecutar → modificar → ver* es la forma más rápida de aprender un concepto nuevo.
6. **Cuenta una historia.** Un `.py` es una caja opaca; un cuaderno te lleva de la mano: contexto → código → resultado → conclusión. Por eso es ideal para enseñar arquitectura, no solo sintaxis.

### El cuidado que exige (y que también enseña buen criterio)

- **El orden de ejecución importa.** Las celdas comparten memoria. Si ejecutas la 5 antes que la 3, puedes usar una variable que aún no existe. Norma: **ejecuta de arriba abajo**. Si te lías, menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **El runtime es temporal.** Si Colab se desconecta, pierdes las variables en memoria (no el cuaderno). Solución: vuelves a ejecutar desde arriba. Por eso todo lo importante lo guardamos **fuera** (BigQuery, Storage), no en memoria.

> **Idea para llevarte:** el cuaderno es para *pensar, aprender y desplegar*. La verdad persistente vive en GCP. Esa separación — exploración efímera vs. infraestructura duradera — es exactamente el modo de trabajo AI-first de esta clase.


In [ ]:
print('¡Hola! Tu cuaderno está ejecutando código.')
import sys
print('Versión de Python:', sys.version.split()[0])


## 0.2 · Python mínimo imprescindible

No necesitas ser programador de Python para esta serie. Solo cuatro ideas:

1. **Variable** = una caja con nombre que guarda un valor.
2. **Diccionario** = pares `clave: valor`. **Es exactamente la forma de JSON** (lo verás en 0.3).
3. **Lista** = una secuencia ordenada de cosas.
4. **Función** = un bloque de código con nombre que puedes reutilizar.

Lee los comentarios (`#`) de la celda y ejecútala.


In [ ]:
# 1) Una variable
ciudad = 'Madrid'
temperatura = 21.5

# 2) Un diccionario (esto ES json): claves entre comillas, valores de cualquier tipo
observacion = {
    'ciudad': ciudad,
    'temp_max': temperatura,
    'lluvia': False,
}

# 3) Una lista de diccionarios (varias observaciones)
observaciones = [observacion, {'ciudad': 'Sevilla', 'temp_max': 30.0, 'lluvia': False}]

# 4) Una función que recibe datos y devuelve algo
def resumen(items):
    nombres = [x['ciudad'] for x in items]   # recorrer la lista y sacar un campo
    return f'Tengo {len(items)} observaciones: {nombres}'

print(resumen(observaciones))


## 0.3 · Qué es una API REST y qué es JSON (en vivo)

- Una **API** es la forma en que un programa le pide datos o acciones a **otro servicio por internet**.
- **REST** es el estilo más común: haces una petición a una **URL** (un *endpoint*) y recibes una respuesta.
- Esa respuesta casi siempre viene en **JSON**: texto con la misma forma que un diccionario de Python.

Vamos a llamar a **Open-Meteo**, una API pública y gratuita de clima (sin clave), y ver el JSON real.


In [ ]:
import requests   # librería para hacer peticiones HTTP, ya viene en Colab

# Endpoint público de Open-Meteo: clima actual de Madrid (lat/lon)
url = 'https://api.open-meteo.com/v1/forecast'
params = {'latitude': 40.4168, 'longitude': -3.7038, 'current_weather': True}

respuesta = requests.get(url, params=params)   # GET = 'dame datos'
print('Código de estado HTTP:', respuesta.status_code, '(200 = OK)')

datos = respuesta.json()   # convertir el texto JSON en un diccionario de Python
datos


In [ ]:
# Navegar dentro del JSON como si fuera un diccionario anidado
clima = datos['current_weather']
print('Temperatura ahora en Madrid:', clima['temperature'], '°C')
print('Viento:', clima['windspeed'], 'km/h')


> **Idea clave:** *toda ingesta batch empieza así* — pides datos a una fuente (una API, una BD, un archivo) y recibes JSON o filas. El Notebook 1 convierte esto en una ingesta real a BigQuery.


## 0.4 · Autenticación: el concepto que asusta y no debería

Para que GCP te deje crear cosas, tiene que **saber quién eres y qué permisos tienes**. Cuatro términos que vas a oír todo el rato:

| Término | Qué es | Analogía |
|---|---|---|
| **Usuario** | Tú, con tu cuenta Google | Tu DNI |
| **Service account (SA)** | Una cuenta-robot para que un programa actúe solo | El carnet de un empleado-robot |
| **Token** | Una credencial temporal que demuestra identidad | Una pulsera de acceso de un día |
| **Secreto** | Un dato sensible (contraseña, API key) guardado a salvo | La llave en una caja fuerte |

**Regla de oro:** las contraseñas y claves **nunca** se escriben en el código. Se guardan en **Secret Manager** y se piden por nombre. Lo usarás desde el Notebook 1.


## 0.5 · Conectar este Colab con tu Google Cloud

Ejecuta la celda. Se abrirá una ventana de Google: elige tu cuenta y acepta. A partir de ahí, `gcloud` y las librerías actúan **como tú** sobre tu proyecto.


In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Autenticado correctamente como tu usuario de Google.')


Ahora dile a `gcloud` **en qué proyecto** trabajar. Cambia `TU_PROJECT_ID` por el ID real de tu proyecto (lo ves en la consola de GCP, arriba). Si aún no tienes proyecto, créalo en https://console.cloud.google.com.


In [ ]:
PROJECT_ID = 'TU_PROJECT_ID'   # <-- CAMBIA ESTO
REGION = 'europe-west1'

import os
os.environ['PROJECT_ID'] = PROJECT_ID
os.environ['REGION'] = REGION

# Fijar el proyecto por defecto en gcloud (el '!' ejecuta un comando de terminal)
!gcloud config set project $PROJECT_ID
print('Proyecto activo configurado.')


In [ ]:
# Tu primer comando real contra GCP: ver la configuración activa
!gcloud config list


## 0.6 · Tu primera consulta a un dataset público de BigQuery

**BigQuery** es el almacén analítico de GCP (la *Capa 2* del embudo). Google publica datasets abiertos gratis. Vamos a contar viajes en el dataset público de **taxis de Nueva York** — sin haber ingerido nada todavía.

Esto demuestra el destino al que todo lo que ingieras acabará llegando.


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

sql = '''
SELECT COUNT(*) AS total_viajes
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2018`
LIMIT 1
'''

fila = list(client.query(sql).result())[0]
print(f'Viajes de taxi amarillo en NYC (2018): {fila.total_viajes:,}')


> Si esto devuelve un número de millones, **tu conexión a GCP funciona de punta a punta**. 🎉


## 0.7 · Los 4 modos de ingesta (el mapa mental de toda la serie)

Cada herramienta de la clase encaja en uno de estos modos. Tenlos siempre presentes:

| Modo | Cuándo | Ejemplo en una frase | Notebook |
|---|---|---|---|
| **Batch periódico** | Datos que no cambian al segundo | "Cada noche, trae el clima de hoy" | 1, 6 |
| **Streaming** | Datos que cambian al segundo | "Procesa cada viaje de taxi al instante" | 3 |
| **CDC** | Espejar una BD sin tocar la app | "Replica mi Postgres a BigQuery" | 5 |
| **Event-driven** | Reaccionar cuando algo pasa | "Cuando suban un archivo, procésalo" | 4 |

Y dos herramientas de **orquestación** que coordinan a las demás: **Workflows** (Notebook 2) y **Pub/Sub** como sistema nervioso (Notebook 3).


## 0.8 · Diagnóstico del entorno (verde / rojo)

Antes de seguir, comprobemos que todo está listo. Esta celda:
- habilita las APIs que usará la serie,
- crea un **dataset de trabajo** y un **bucket** de trabajo (si no existen),
- y te da un resumen verde/rojo.

Es idempotente: puedes ejecutarla varias veces sin romper nada.


In [ ]:
# Habilitar las APIs de toda la serie (tarda ~1 min la primera vez)
!gcloud services enable \
  bigquery.googleapis.com storage.googleapis.com pubsub.googleapis.com \
  eventarc.googleapis.com workflows.googleapis.com run.googleapis.com \
  cloudfunctions.googleapis.com secretmanager.googleapis.com \
  cloudscheduler.googleapis.com datastream.googleapis.com \
  --project=$PROJECT_ID
print('APIs habilitadas.')


In [ ]:
# Crear dataset y bucket de trabajo (idempotente)
DATASET = 'capa1_trabajo'
BUCKET = f'{PROJECT_ID}-capa1-trabajo'

# Dataset BigQuery
!bq --location=EU mk --dataset --force $PROJECT_ID:$DATASET 2>/dev/null; echo 'dataset ok'
# Bucket GCS
!gcloud storage buckets create gs://$BUCKET --location=EU --uniform-bucket-level-access 2>/dev/null; echo 'bucket ok (o ya existía)'


In [ ]:
# Resumen verde/rojo del entorno
import subprocess

def check(nombre, comando):
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    ok = r.returncode == 0
    print(('✅' if ok else '❌'), nombre)
    return ok

print('--- Diagnóstico del entorno ---')
check('Proyecto activo', f'gcloud config get-value project | grep -q {PROJECT_ID}')
check('Acceso a BigQuery', f'bq query --use_legacy_sql=false --project_id={PROJECT_ID} "SELECT 1"')
check('Dataset de trabajo', f'bq show {PROJECT_ID}:{DATASET}')
check('Bucket de trabajo', f'gcloud storage ls gs://{BUCKET}')
print('\nSi todo está en verde, estás listo para el Notebook 1.')


## 0.9 · Bonus — Tu primer prompt para Claude Code + gcloud

En el resto de la serie, además de las celdas de Colab, usarás **Claude Code** (en una terminal con `gcloud` autenticado) para **desplegar infraestructura completa de una vez**. La IA genera, revisa y *ejecuta* los comandos.

Este es el patrón. Copia este prompt en Claude Code para preparar el entorno desde la terminal (equivalente a la celda 0.8):

```
Eres un ingeniero senior de GCP con gcloud autenticado en el proyecto <TU_PROJECT_ID>.

OBJETIVO
1. Habilita estas APIs: bigquery, storage, pubsub, eventarc, workflows, run,
   cloudfunctions, secretmanager, cloudscheduler, datastream.
2. Crea el dataset BigQuery 'capa1_trabajo' en location EU (si no existe).
3. Crea el bucket 'gs://<TU_PROJECT_ID>-capa1-trabajo' en EU con acceso uniforme.

EJECUCIÓN
- Ejecuta los comandos gcloud tú mismo, uno a uno, y muéstrame la salida.
- Antes de cualquier comando destructivo, párate y pídeme confirmación.
- Al final, dame un comando de verificación.
```

> La diferencia con un prompt normal: la sección **EJECUCIÓN**. Como Claude Code *ejecuta de verdad*, le pedimos que confirme antes de destruir, que muestre salidas y que verifique al final.


## Checklist de salida

Antes de pasar al Notebook 1, confirma:

- [ ] La celda de autenticación corrió sin error.
- [ ] `gcloud config list` muestra tu proyecto.
- [ ] La query al dataset público de taxis devolvió millones de viajes.
- [ ] El diagnóstico 0.8 está **todo en verde**.
- [ ] Entiendes, con tus palabras, los 4 modos de ingesta.

**Siguiente:** `01_batch_api_a_bigquery.ipynb` — tu primera ingesta batch real (API pública → BigQuery, idempotente).
